## 1. Load and Clean Source Data

### 1.1 Load and Deduplicate Overton Data

The Overton publication export is loaded and duplicate publication
records are removed using normalized DOI identifiers. DOI strings are
standardized before duplicate detection to account for differences in
capitalization, whitespace, and DOI URL prefixes.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

DATA_DIR = Path("../data")

OVERTON_FILE = DATA_DIR / "overton_full.xlsx"

overton = pd.read_excel(
    OVERTON_FILE
)

print("Overton file loaded.")
print(f"Rows    : {len(overton):,}")
print(f"Columns : {len(overton.columns):,}")

print("\nColumns:")
print(overton.columns.tolist())
print()

Overton file loaded.
Rows    : 21,074
Columns : 19

Columns:
['Authors', 'Author full names', 'Author(s) ID', 'Title', 'Year', 'Source title', 'Cited by', 'DOI', 'Link', 'Affiliations', 'Authors with affiliations', 'Abstract', 'Author Keywords', 'Index Keywords', 'Funding Details', 'Funding Texts', 'Publisher', 'Document Type', 'Source']



In [4]:
def normalize_doi(value):
    """
    Normalize DOI values for reliable matching and duplicate detection.
    """

    if pd.isna(value):
        return pd.NA

    doi = str(value).strip().lower()

    # Remove common DOI URL/prefix forms.
    doi = re.sub(
        r"^https?://(?:dx\.)?doi\.org/",
        "",
        doi,
    )

    doi = re.sub(
        r"^doi:\s*",
        "",
        doi,
    )

    doi = doi.strip()

    if not doi:
        return pd.NA

    return doi


overton["DOI_normalized"] = (
    overton["DOI"]
    .apply(normalize_doi)
)

print(
    "Rows                  :",
    f"{len(overton):,}"
)

print(
    "Rows with DOI         :",
    f"{overton['DOI_normalized'].notna().sum():,}"
)

print(
    "Rows without DOI      :",
    f"{overton['DOI_normalized'].isna().sum():,}"
)

print(
    "Unique normalized DOIs:",
    f"{overton['DOI_normalized'].nunique():,}"
)

Rows                  : 21,074
Rows with DOI         : 21,074
Rows without DOI      : 0
Unique normalized DOIs: 14,065


In [5]:
overton_doi_duplicates = (
    overton[
        overton["DOI_normalized"].notna()
        & overton["DOI_normalized"].duplicated(
            keep=False
        )
    ]
    .sort_values(
        "DOI_normalized"
    )
    .copy()
)

duplicate_doi_counts = (
    overton_doi_duplicates[
        "DOI_normalized"
    ]
    .value_counts()
)

print(
    "Unique DOIs appearing more than once:",
    f"{len(duplicate_doi_counts):,}"
)

print(
    "Rows involved in DOI duplicates:",
    f"{len(overton_doi_duplicates):,}"
)

print(
    "Duplicate rows removable:",
    f"{overton['DOI_normalized'].duplicated().sum():,}"
)

duplicate_doi_counts.head(20)

Unique DOIs appearing more than once: 5,030
Rows involved in DOI duplicates: 12,039
Duplicate rows removable: 7,009


DOI_normalized
10.1007/978-3-0348-0133-1             62
10.1201/9781315371290                 51
10.4324/9780203928325                 31
10.4324/9780203928912                 24
10.4324/9781315770734                 20
10.4324/9781315769356                 18
10.4337/9781783473335                 16
10.4324/9781315766607                 14
10.4324/9781315778914                 13
10.4324/9781315609546                 10
10.1007/978-3-319-16649-0              8
10.1002/9781118562581                  6
10.1016/b978-0-12-819727-1.00077-7     6
10.1109/37.969131                      6
10.1190/1.1778241                      6
10.17226/25483                         6
10.2118/210229-ms                      6
10.3233/ssw210025                      6
10.1002/2014jd022335                   4
10.1002/9781119196082                  4
Name: count, dtype: int64

In [6]:
# Check whether rows sharing a DOI also share the same title.

doi_title_check = (
    overton[
        overton["DOI_normalized"].notna()
    ]
    .groupby("DOI_normalized")
    .agg(
        row_count=("DOI_normalized", "size"),
        unique_titles=("Title", "nunique"),
    )
    .reset_index()
)

duplicate_doi_title_check = (
    doi_title_check[
        doi_title_check["row_count"] > 1
    ]
    .copy()
)

conflicting_doi_titles = (
    duplicate_doi_title_check[
        duplicate_doi_title_check[
            "unique_titles"
        ] > 1
    ]
)

print(
    "Duplicated DOIs:",
    f"{len(duplicate_doi_title_check):,}"
)

print(
    "Duplicated DOIs with one title:",
    f"{(duplicate_doi_title_check['unique_titles'] == 1).sum():,}"
)

print(
    "Duplicated DOIs with multiple titles:",
    f"{len(conflicting_doi_titles):,}"
)

conflicting_doi_titles.head(20)

Duplicated DOIs: 5,030
Duplicated DOIs with one title: 4,987
Duplicated DOIs with multiple titles: 43


,DOI_normalized,row_count,unique_titles
49,10.1002/9781118562581,6,2
58,10.1002/9781119196082,4,2
68,10.1002/9781444319514,4,2
541,10.1007/978-3-0348-0133-1,62,31
567,10.1007/978-3-319-16649-0,8,8
577,10.1007/978-3-319-21732-1,2,2
1343,10.1007/s13171-013-0027-y,2,2
1360,10.1007/s13571-013-0071-6,2,2
1549,10.1016/b978-0-12-819727-1.00077-7,6,2
4180,10.1016/j.proeng.2015.01.510,2,2


In [7]:
conflicting_doi_rows = (
    overton[
        overton["DOI_normalized"].isin(
            conflicting_doi_titles["DOI_normalized"]
        )
    ][
        [
            "DOI_normalized",
            "Title",
            "Year",
            "Source title",
            "Document Type",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["DOI_normalized", "Title"]
    )
    .reset_index(drop=True)
)

print(
    "Conflicting DOI groups:",
    conflicting_doi_rows[
        "DOI_normalized"
    ].nunique()
)

print(
    "Distinct DOI-title combinations:",
    len(conflicting_doi_rows)
)

conflicting_doi_rows

Conflicting DOI groups: 43
Distinct DOI-title combinations: 254


,DOI_normalized,Title,Year,Source title,Document Type
0,10.1002/9781118562581,Foreword,2013,Smart Grids,Editorial
1,10.1002/9781118562581,Smart Grids,2013,Smart Grids,Book
2,10.1002/9781119196082,Preface,2018,The Chemistry of Membranes Used in Fuel Cells:...,Editorial
3,10.1002/9781119196082,The chemistry of membranes used in fuel cells:...,2018,The Chemistry of Membranes Used in Fuel Cells:...,Book
4,10.1002/9781444319514,"The Rise of the Network Society, Second Editio...",2010,"The Rise of the Network Society, Second Editio...",Book
...,...,...,...,...,...
249,10.4337/9781783473335,Rail economics and regulation,2015,"Rail Economics, Policy and Regulation in Europe",Book chapter
250,10.4337/9781783473335,"Rail economics, policy and regulation in Europe",2015,"Rail Economics, Policy and Regulation in Europe",Book
251,10.4337/9781783473335,Railways and demographic change,2015,"Rail Economics, Policy and Regulation in Europe",Book chapter
252,10.4337/9781783473335,Rolling stock companies (Roscos): Experience f...,2015,"Rail Economics, Policy and Regulation in Europe",Book chapter


In [9]:
def normalize_title(value):
    """
    Normalize publication titles for duplicate detection.
    """

    if pd.isna(value):
        return pd.NA

    title = str(value).lower().strip()

    # Normalize whitespace
    title = re.sub(r"\s+", " ", title).strip()

    if not title:
        return pd.NA

    return title


overton["Title_normalized"] = (
    overton["Title"]
    .apply(normalize_title)
)

# Rows for which both DOI and title are available.
has_doi_title = (
    overton["DOI_normalized"].notna()
    & overton["Title_normalized"].notna()
)

# Duplicate DOI + title combinations.
doi_title_duplicate_mask = (
    has_doi_title
    & overton.duplicated(
        subset=[
            "DOI_normalized",
            "Title_normalized",
        ],
        keep=False,
    )
)

doi_title_duplicates = (
    overton[
        doi_title_duplicate_mask
    ]
    .copy()
)

# Number of extra rows that would be removed by keeping
# one occurrence of each DOI + title combination.
duplicate_rows_removable = (
    overton.loc[
        has_doi_title
    ]
    .duplicated(
        subset=[
            "DOI_normalized",
            "Title_normalized",
        ],
        keep="first",
    )
    .sum()
)

# Number of unique DOI + title publication records.
unique_doi_title_publications = (
    overton.loc[
        has_doi_title,
        [
            "DOI_normalized",
            "Title_normalized",
        ],
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Rows with DOI + title:",
    f"{has_doi_title.sum():,}"
)

print(
    "Rows involved in DOI+title duplicates:",
    f"{len(doi_title_duplicates):,}"
)

print(
    "Duplicate rows removable by DOI+title:",
    f"{duplicate_rows_removable:,}"
)

print(
    "Unique DOI+title publications:",
    f"{unique_doi_title_publications:,}"
)

Rows with DOI + title: 21,074
Rows involved in DOI+title duplicates: 11,879
Duplicate rows removable by DOI+title: 6,808
Unique DOI+title publications: 14,266


In [10]:
# Deduplicate Overton publications using normalized DOI + title.
# Keep the first occurrence of each unique publication.

overton_clean = (
    overton
    .drop_duplicates(
        subset=[
            "DOI_normalized",
            "Title_normalized",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print(f"Original Overton rows : {len(overton):,}")
print(f"Duplicate rows removed: {len(overton) - len(overton_clean):,}")
print(f"Unique publications   : {len(overton_clean):,}")

# Verify uniqueness.
remaining_duplicates = (
    overton_clean
    .duplicated(
        subset=[
            "DOI_normalized",
            "Title_normalized",
        ]
    )
    .sum()
)

print(
    "Remaining DOI+title duplicates:",
    remaining_duplicates
)

Original Overton rows : 21,074
Duplicate rows removed: 6,808
Unique publications   : 14,266
Remaining DOI+title duplicates: 0


In [12]:
# Examine DOI uniqueness after DOI+title deduplication.

doi_counts_clean = (
    overton_clean["DOI_normalized"]
    .value_counts()
)

shared_dois_clean = (
    doi_counts_clean[
        doi_counts_clean > 1
    ]
)

rows_with_shared_doi = (
    overton_clean[
        overton_clean["DOI_normalized"].isin(
            shared_dois_clean.index
        )
    ]
)

print(f"Unique publication records : {len(overton_clean):,}")
print(
    f"Unique DOI values          : "
    f"{overton_clean['DOI_normalized'].nunique():,}"
)
print(
    f"DOIs linked to >1 title    : "
    f"{len(shared_dois_clean):,}"
)
print(
    f"Records using shared DOIs  : "
    f"{len(rows_with_shared_doi):,}"
)

shared_dois_clean.head(20)

Unique publication records : 14,266
Unique DOI values          : 14,065
DOIs linked to >1 title    : 33
Records using shared DOIs  : 234


DOI_normalized
10.4324/9780203928325           31
10.1007/978-3-0348-0133-1       31
10.4324/9781315770734           20
10.4324/9781315769356           18
10.1201/9781315371290           17
10.4337/9781783473335           16
10.4324/9781315766607           14
10.4324/9781315778914           13
10.4324/9780203928912           12
10.4324/9781315609546           10
10.1007/978-3-319-16649-0        8
10.1145/2676726.2677005          2
10.2514/6.2016-2493              2
10.1007/978-3-319-21732-1        2
10.1111/gcb.14606                2
10.17226/26092                   2
10.1002/9781444319514            2
10.1016/j.proeng.2015.01.510     2
10.3390/su11195286               2
10.1007/s13571-013-0071-6        2
Name: count, dtype: int64

### 1.2 Load and Inspect Scopus Data

The Scopus export is loaded separately and inspected before
deduplication and matching with the Overton publication records.

In [14]:
SCOPUS_FILE = DATA_DIR / "scopus_full.xlsx"

scopus = pd.read_excel(
    SCOPUS_FILE
)

print("Scopus file loaded.")
print(f"Rows    : {len(scopus):,}")
print(f"Columns : {len(scopus.columns):,}")

print("\nColumns:")
print(scopus.columns.tolist())
print()

Scopus file loaded.
Rows    : 16,409
Columns : 17

Columns:
['Link', 'Authors', 'Author full names', 'Author(s) ID', 'Title', 'Year', 'Source title', 'Cited by', 'Affiliations', 'Publisher', 'Abbreviated Source Title', 'DOI', 'Abstract', 'Author Keywords', 'Index Keywords', 'Funding Details', 'Funding Texts']



In [15]:
# Normalize Scopus DOIs using the same rule used for Overton.

scopus["DOI_normalized"] = (
    scopus["DOI"]
    .apply(normalize_doi)
)

print(f"Total Scopus rows       : {len(scopus):,}")
print(
    f"Rows with DOI           : "
    f"{scopus['DOI_normalized'].notna().sum():,}"
)
print(
    f"Rows without DOI        : "
    f"{scopus['DOI_normalized'].isna().sum():,}"
)
print(
    f"Unique normalized DOIs  : "
    f"{scopus['DOI_normalized'].nunique():,}"
)

Total Scopus rows       : 16,409
Rows with DOI           : 16,409
Rows without DOI        : 0
Unique normalized DOIs  : 15,858


In [16]:
scopus_duplicate_dois = (
    scopus["DOI_normalized"]
    .value_counts()
)

scopus_duplicate_dois = (
    scopus_duplicate_dois[
        scopus_duplicate_dois > 1
    ]
)

scopus_rows_with_duplicate_doi = (
    scopus[
        scopus["DOI_normalized"].isin(
            scopus_duplicate_dois.index
        )
    ]
)

print(
    "DOIs appearing more than once :",
    f"{len(scopus_duplicate_dois):,}"
)

print(
    "Rows involved in duplicates   :",
    f"{len(scopus_rows_with_duplicate_doi):,}"
)

print(
    "Extra DOI rows                :",
    f"{int(scopus_duplicate_dois.sub(1).sum()):,}"
)

scopus_duplicate_dois.head(20)

DOIs appearing more than once : 7
Rows involved in duplicates   : 558
Extra DOI rows                : 551


DOI_normalized
0                                     546
10.23919/pcmp.2024.000216               2
10.12989/scs.2022.45.2.281              2
10.12989/scs.2022.45.2.205              2
10.1111/cgf.14080                       2
10.1016/b978-0-12-811968-6.00010-3      2
10.1016/b978-0-12-811968-6.00009-7      2
Name: count, dtype: int64

In [17]:
# Convert empty normalized DOI strings to missing values.

scopus["DOI_normalized"] = (
    scopus["DOI_normalized"]
    .replace("", pd.NA)
)

print(f"Total Scopus rows      : {len(scopus):,}")
print(
    f"Rows with DOI          : "
    f"{scopus['DOI_normalized'].notna().sum():,}"
)
print(
    f"Rows without DOI       : "
    f"{scopus['DOI_normalized'].isna().sum():,}"
)
print(
    f"Unique DOI values      : "
    f"{scopus['DOI_normalized'].nunique():,}"
)

Total Scopus rows      : 16,409
Rows with DOI          : 16,409
Rows without DOI       : 0
Unique DOI values      : 15,858


In [19]:
# Treat DOI "0" as missing.

scopus["DOI_normalized"] = (
    scopus["DOI_normalized"]
    .replace("0", pd.NA)
)

print(f"Total Scopus rows     : {len(scopus):,}")
print(
    f"Rows with DOI         : "
    f"{scopus['DOI_normalized'].notna().sum():,}"
)
print(
    f"Rows without DOI      : "
    f"{scopus['DOI_normalized'].isna().sum():,}"
)
print(
    f"Unique valid DOIs     : "
    f"{scopus['DOI_normalized'].nunique():,}"
)

Total Scopus rows     : 16,409
Rows with DOI         : 15,863
Rows without DOI      : 546
Unique valid DOIs     : 15,857


In [21]:
# Inspect genuine repeated DOI records in Scopus.

scopus_duplicate_dois = (
    scopus.loc[
        scopus["DOI_normalized"].notna(),
        "DOI_normalized",
    ]
    .value_counts()
)

scopus_duplicate_dois = (
    scopus_duplicate_dois[
        scopus_duplicate_dois > 1
    ]
)

scopus_duplicate_rows = (
    scopus[
        scopus["DOI_normalized"].isin(
            scopus_duplicate_dois.index
        )
    ][
        [
            "DOI_normalized",
            "Title",
            "Year",
            "Source title",
        ]
    ]
    .sort_values(
        ["DOI_normalized", "Title"]
    )
    .reset_index(drop=True)
)

print(
    "Genuine duplicated DOIs:",
    len(scopus_duplicate_dois)
)

print(
    "Rows involved:",
    len(scopus_duplicate_rows)
)

scopus_duplicate_rows

Genuine duplicated DOIs: 6
Rows involved: 12


,DOI_normalized,Title,Year,Source title
0,10.1016/b978-0-12-811968-6.00009-7,Time-Series Classification Methods: Review and...,2018,Big Data Application in Power Systems
1,10.1016/b978-0-12-811968-6.00009-7,Time-Series Classification Methods: Review and...,2017,Big Data Application in Power Systems
2,10.1016/b978-0-12-811968-6.00010-3,Future Trends for Big Data Application in Powe...,2018,Big Data Application in Power Systems
3,10.1016/b978-0-12-811968-6.00010-3,Future Trends for Big Data Application in Powe...,2017,Big Data Application in Power Systems
4,10.1111/cgf.14080,EGGS: Sparsity-Specific Code Generation,2020,Computer Graphics Forum
5,10.1111/cgf.14080,EGGS: Sparsity-Specific Code Generation,2020,Eurographics Symposium on Geometry Processing
6,10.12989/scs.2022.45.2.205,Ensembles of neural network with stochastic op...,2022,Steel and Composite Structures
7,10.12989/scs.2022.45.2.205,Ensembles of neural network with stochastic op...,2022,Structural Engineering and Mechanics
8,10.12989/scs.2022.45.2.281,ANN-Incorporated satin bowerbird optimizer for...,2022,Steel and Composite Structures
9,10.12989/scs.2022.45.2.281,ANN-Incorporated satin bowerbird optimizer for...,2022,Structural Engineering and Mechanics


In [22]:
# Deduplicate Scopus records with valid DOIs.
# Keep the first occurrence of each DOI.
# Records without DOI are retained individually.

scopus_with_doi = (
    scopus[
        scopus["DOI_normalized"].notna()
    ]
    .drop_duplicates(
        subset="DOI_normalized",
        keep="first",
    )
)

scopus_without_doi = (
    scopus[
        scopus["DOI_normalized"].isna()
    ]
)

scopus_clean = (
    pd.concat(
        [
            scopus_with_doi,
            scopus_without_doi,
        ],
        ignore_index=True,
    )
)

print(f"Original Scopus rows : {len(scopus):,}")
print(f"Rows with valid DOI  : {len(scopus_with_doi):,}")
print(f"Rows without DOI     : {len(scopus_without_doi):,}")
print(f"Clean Scopus rows    : {len(scopus_clean):,}")
print(
    f"Rows removed         : "
    f"{len(scopus) - len(scopus_clean):,}"
)

remaining_doi_duplicates = (
    scopus_clean.loc[
        scopus_clean["DOI_normalized"].notna(),
        "DOI_normalized",
    ]
    .duplicated()
    .sum()
)

print(
    "Remaining valid-DOI duplicates:",
    remaining_doi_duplicates
)

Original Scopus rows : 16,409
Rows with valid DOI  : 15,857
Rows without DOI     : 546
Clean Scopus rows    : 16,403
Rows removed         : 6
Remaining valid-DOI duplicates: 0


In [23]:
# Save cleaned Overton and Scopus datasets separately.

OVERTON_CLEAN_FILE = (
    DATA_DIR / "overton_full_clean.xlsx"
)

SCOPUS_CLEAN_FILE = (
    DATA_DIR / "scopus_full_clean.xlsx"
)

overton_clean.to_excel(
    OVERTON_CLEAN_FILE,
    index=False,
)

scopus_clean.to_excel(
    SCOPUS_CLEAN_FILE,
    index=False,
)

print("Saved cleaned datasets:")
print(
    f"Overton : {OVERTON_CLEAN_FILE} "
    f"({len(overton_clean):,} rows)"
)
print(
    f"Scopus  : {SCOPUS_CLEAN_FILE} "
    f"({len(scopus_clean):,} rows)"
)

Saved cleaned datasets:
Overton : data\overton_full_clean.xlsx (14,266 rows)
Scopus  : data\scopus_full_clean.xlsx (16,403 rows)
